# FEDS: Run server + clients (Colab)

Quick check: run FEDS server and MNIST clients. Artifacts are created in the next cell if missing. Server uses 300 rounds by default; interrupt after a few rounds for a quick check.

In [ ]:
# Clone or set path (Colab: clone; local: skip and set REPO)
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Replace with your repo URL if you prefer clone; else upload and set REPO to current dir
    !git clone https://github.com/masud1901/adaptive-sparsification.git 2>/dev/null || true
    REPO = "/content/adaptive-sparsification"
    if not os.path.exists(REPO):
        REPO = os.getcwd()
else:
    REPO = os.path.abspath(os.getcwd())

os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "utils"))
print("REPO:", REPO)

In [ ]:
# Create initial model and K pickle if missing
import subprocess
r = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "create_artifacts_colab.py")], cwd=REPO, capture_output=True, text=True)
print(r.stdout or r.stderr or "OK")

In [ ]:
# Install deps (Colab has torch; may need flwr tensorboard)
!pip install -q flwr tensorboard 2>/dev/null
import flwr
print("flwr:", flwr.__version__)

In [ ]:
# Run server in background (small rounds for quick check)
import threading
import subprocess

env = os.environ.copy()
env["PYTHONPATH"] = os.path.join(REPO, "utils")
env["NUM_ROUNDS"] = "3"

def run_server():
    subprocess.run([sys.executable, os.path.join(REPO, "server.py")], cwd=REPO, env=env)

t = threading.Thread(target=run_server)
t.daemon = True
t.start()
import time
time.sleep(8)
print("Server started.")

In [ ]:
# Run 10 MNIST clients
client_script = os.path.join(REPO, "clients", "client-MNIST.py")
procs = []
for i in range(10):
    p = subprocess.Popen([sys.executable, client_script, "--seed", str(i)], cwd=REPO, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    procs.append(p)
for p in procs:
    p.wait()
print("Clients finished.")

In [ ]:
# Check FEDS outputs
import json
k_tracker = os.path.join(REPO, "feds_k_tracker.json")
if os.path.exists(k_tracker):
    with open(k_tracker) as f:
        print(json.dumps(json.load(f), indent=2)[:1500])
else:
    print("No feds_k_tracker.json yet")
runs = os.path.join(REPO, "runs")
if os.path.isdir(runs):
    print("TensorBoard runs:", os.listdir(runs))